# Algorithmic Trading with Python & Google Colab
## Session 3: From Notebook to Trading System: Cloud Deployment, Monitoring & Live Simulation

*(c) Dr. Yves J. Hilpisch | The Python Quants GmbH | https://tpq.io | https://hilpisch.com*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yveshilpisch/pyalgo/blob/main/notebooks/03_cloud_deployment_monitoring.ipynb)

---

### Objectives
1. **Architectural Separation**: Loading serialized PyTorch model weights (`best_trading_dnn.pt`) onto a lightweight CPU execution runtime.
2. **Real-Time ZeroMQ Streaming**: Publishing market ticks over ZeroMQ (`tcp://127.0.0.1:5555`) and consuming push events via SUB sockets.
3. **Transactional SQLite Persistence**: Decoupled database recording for tick audit logs and portfolio telemetry.
4. **Live Trading Engine & Circuit Breakers**: Generating real-time inferences, managing position transitions, and enforcing risk halts.
5. **Real-Time Monitoring Dashboard**: Live visual telemetry of NAV equity growth, drawdown gauges, and probability distributions.
6. **Path to Cloud Production**: Transitioning from Colab sandbox to containerized VPS deployments.


In [ ]:
import time
import threading
import sqlite3
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zmq
import torch
import torch.nn as nn
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("Session 3 Deployment Environment Initialized.")


## 1. Minimal Production Model Architecture & Loading

In production, model inference operates on standard CPU without requiring costly GPU instances.


In [ ]:
class ProductionDNN(nn.Module):
    def __init__(self, input_dim=7, hidden_units=[64, 32]):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_units:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

prod_model = ProductionDNN(input_dim=7, hidden_units=[64, 32])

if Path('best_trading_dnn.pt').exists():
    prod_model.load_state_dict(torch.load('best_trading_dnn.pt', map_location='cpu'))
    print("Successfully loaded trained model artifact 'best_trading_dnn.pt'.")
else:
    print("Pretrained artifact not found locally; running with initialized weights for demonstration.")

prod_model.eval()


## 2. ZeroMQ Real-Time Streaming Architecture

We use **ZeroMQ (ZMQ)** for asynchronous, low-latency market data transport:
- **Publisher (PUB)**: Broadcasts real-time JSON ticks over `tcp://127.0.0.1:5555`.
- **Database Subscriber (SUB)**: Persists raw tick history into `ticks.db`.
- **Trading Engine Subscriber (SUB)**: Evaluates live signals, calculates drawdown limits, and executes simulated orders.


In [ ]:
def run_tick_server(
    bind_addr: str = "tcp://127.0.0.1:5555",
    symbol: str = "SPY",
    start_price: float = 450.0,
    dt: float = 0.01,
    sigma: float = 0.0005,
    max_ticks: int = 120
):
    ctx = zmq.Context()
    socket = ctx.socket(zmq.PUB)
    socket.setsockopt(zmq.LINGER, 0)
    socket.bind(bind_addr)
    price = start_price
    rng = np.random.default_rng()
    for _ in range(max_ticks):
        shock = rng.normal(0.0, sigma * price)
        price = max(1.0, price + shock)
        payload = {
            "time": datetime.now(timezone.utc).isoformat(),
            "symbol": symbol,
            "price": round(float(price), 4)
        }
        try:
            socket.send_json(payload)
        except (zmq.ContextTerminated, zmq.ZMQError):
            break
        time.sleep(dt)
    socket.close(0)
    ctx.term()

# Start ZeroMQ Market Publisher in a background thread
server_thread = threading.Thread(
    target=run_tick_server,
    kwargs={'bind_addr': 'tcp://127.0.0.1:5555', 'symbol': 'SPY', 'start_price': 450.0, 'dt': 0.01, 'max_ticks': 120},
    daemon=True
)
server_thread.start()
print("ZeroMQ Tick Server is broadcasting live market events on tcp://127.0.0.1:5555...")


## 3. Real-Time Trading Client & Live Inference

The `ZMQTradingClient` subscribes to the live ZeroMQ feed, computes rolling features, performs PyTorch model inferences, enforces circuit breakers, and persists all trades to SQLite.


In [ ]:
class ZMQTradingClient:
    def __init__(
        self,
        connect_addr: str = "tcp://127.0.0.1:5555",
        model: nn.Module = prod_model,
        initial_capital: float = 100_000.0,
        max_drawdown_limit: float = 0.10,
        tc_rate: float = 0.0005,
        db_path: str = "webinar_live_trading.db"
    ):
        self.connect_addr = connect_addr
        self.model = model.eval()
        self.cash = initial_capital
        self.nav = initial_capital
        self.peak_nav = initial_capital
        self.position = 0
        self.units_per_trade = 100
        self.max_dd_limit = max_drawdown_limit
        self.tc_rate = tc_rate
        self.halted = False
        self.prices = []
        self.timestamps = []
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        with self.conn:
            self.conn.execute("CREATE TABLE IF NOT EXISTS ticks (id INTEGER PRIMARY KEY AUTOINCREMENT, timestamp TEXT, symbol TEXT, price REAL)")
            self.conn.execute("CREATE TABLE IF NOT EXISTS signals (id INTEGER PRIMARY KEY AUTOINCREMENT, timestamp TEXT, symbol TEXT, prob_up REAL, signal INTEGER)")
            self.conn.execute("CREATE TABLE IF NOT EXISTS orders (id INTEGER PRIMARY KEY AUTOINCREMENT, timestamp TEXT, symbol TEXT, side TEXT, units INTEGER, price REAL, cost REAL)")
            self.conn.execute("CREATE TABLE IF NOT EXISTS portfolio_state (id INTEGER PRIMARY KEY AUTOINCREMENT, timestamp TEXT, position INTEGER, cash REAL, nav REAL, drawdown REAL)")

    def process_tick(self, timestamp: str, symbol: str, price: float):
        self.prices.append(price)
        self.timestamps.append(timestamp)
        with self.conn:
            self.conn.execute("INSERT INTO ticks (timestamp, symbol, price) VALUES (?, ?, ?)", (timestamp, symbol, price))
        if len(self.prices) < 25 or self.halted:
            return
        p_arr = np.array(self.prices[-25:])
        rets = np.diff(np.log(p_arr))
        feat_vector = np.concatenate([rets[-5:][::-1], [np.std(rets[-20:]), np.mean(rets[-10:])]])
        x_t = torch.tensor(feat_vector, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            prob = torch.sigmoid(self.model(x_t)).item()
        target_pos = 1 if prob > 0.52 else (-1 if prob < 0.48 else 0)
        with self.conn:
            self.conn.execute("INSERT INTO signals (timestamp, symbol, prob_up, signal) VALUES (?, ?, ?, ?)", (timestamp, symbol, prob, target_pos))
        delta = target_pos - self.position
        if delta != 0:
            units = abs(delta) * self.units_per_trade
            side = "BUY" if delta > 0 else "SELL"
            cost = units * price * self.tc_rate
            self.cash -= (delta * self.units_per_trade * price + cost)
            self.position = target_pos
            with self.conn:
                self.conn.execute("INSERT INTO orders (timestamp, symbol, side, units, price, cost) VALUES (?, ?, ?, ?, ?, ?)", (timestamp, symbol, side, units, price, cost))
        self.nav = self.cash + (self.position * self.units_per_trade * price)
        if self.nav > self.peak_nav:
            self.peak_nav = self.nav
        dd = (self.peak_nav - self.nav) / self.peak_nav if self.peak_nav > 0 else 0.0
        if dd >= self.max_dd_limit:
            self.halted = True
            print(f"[{timestamp}] CIRCUIT BREAKER TRIPPED! Drawdown: {dd:.2%}")
        with self.conn:
            self.conn.execute("INSERT INTO portfolio_state (timestamp, position, cash, nav, drawdown) VALUES (?, ?, ?, ?, ?)", (timestamp, self.position, self.cash, self.nav, dd))

    def run(self, max_ticks: int = 100):
        ctx = zmq.Context()
        socket = ctx.socket(zmq.SUB)
        socket.setsockopt(zmq.LINGER, 0)
        socket.setsockopt(zmq.RCVTIMEO, 500)
        socket.connect(self.connect_addr)
        socket.setsockopt_string(zmq.SUBSCRIBE, "")
        count = 0
        idle_timeouts = 0
        while count < max_ticks:
            try:
                tick = socket.recv_json()
                idle_timeouts = 0
            except zmq.Again:
                idle_timeouts += 1
                if idle_timeouts >= 3:
                    break
                continue
            except (zmq.ContextTerminated, zmq.ZMQError):
                break
            self.process_tick(tick["time"], tick["symbol"], float(tick["price"]))
            count += 1
            if count % 20 == 0:
                print(f"[{tick['time'][:19]}] Ticks: {count:03d} | Price: {tick['price']:.2f} | Pos: {self.position:+d} | NAV: ${self.nav:,.2f}")
        socket.close(0)
        self.conn.close()
        ctx.term()

# Connect Trading Engine client to the live stream
client = ZMQTradingClient(
    connect_addr='tcp://127.0.0.1:5555',
    model=prod_model,
    initial_capital=100000.0,
    max_drawdown_limit=0.10,
    db_path="webinar_live_trading.db"
)

# Consume 100 live stream ticks
client.run(max_ticks=100)


## 4. Live Dashboard & Database Auditing

We query SQLite to inspect executed orders, verify signal probabilities, and render the real-time operational dashboard.


In [ ]:
conn = sqlite3.connect("webinar_live_trading.db")

orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
portfolio_df = pd.read_sql_query("SELECT * FROM portfolio_state", conn)
signals_df = pd.read_sql_query("SELECT * FROM signals", conn)

print("--- AUDIT: LAST 5 EXECUTED ORDERS ---")
print(orders_df.tail())

# Dashboard Visualizations
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)

# 1. NAV Growth
axes[0].plot(portfolio_df.index, portfolio_df['nav'], color='#27AE60', linewidth=2, label='Portfolio NAV ($)')
axes[0].axhline(100000, color='gray', linestyle='--', label='Initial Capital ($100k)')
axes[0].set_title('Live Monitoring: Portfolio NAV Evolution')
axes[0].set_ylabel('NAV ($)')
axes[0].legend()

# 2. Drawdown Gauge
axes[1].fill_between(portfolio_df.index, -portfolio_df['drawdown'] * 100, 0, color='#EB5757', alpha=0.4, label='Drawdown (%)')
axes[1].axhline(-10, color='red', linestyle='--', label='Circuit Breaker Limit (-10%)')
axes[1].set_title('Live Monitoring: Portfolio Drawdown Gauge')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend()

# 3. Model Prediction Distribution
axes[2].hist(signals_df['prob_up'], bins=30, color='#2F80ED', alpha=0.7, edgecolor='black')
axes[2].axvline(0.52, color='green', linestyle='--', label='Long Threshold (0.52)')
axes[2].axvline(0.48, color='red', linestyle='--', label='Short Threshold (0.48)')
axes[2].set_title('Live Monitoring: Model Prediction Probability Distribution')
axes[2].set_xlabel('Predicted Probability P(Up)')
axes[2].legend()

plt.tight_layout()
plt.show()

conn.close()


---
### Full Series Summary
- **Session 1 (Discover)**: Efficient markets, random walks, statistical tests, and linear baseline backtesting.
- **Session 2 (Learn)**: PyTorch GPU training, deep neural architectures, and confidence thresholding.
- **Session 3 (Deploy)**: ZeroMQ streaming server/client architecture, SQLite persistence, and operational risk management.
